In [1]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import sys

# Packages are loaded via PYSPARK_SUBMIT_ARGS set in compose.yml.
# pyspark-notebook:2025-12-31 ships Spark 4.1.0  print spark.version to confirm.

spark = (
    SparkSession.builder
    .appName("project2")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")

    # Iceberg
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # Catalog named 'lakehouse'  use it as: lakehouse.<database>.<table>
    .config("spark.sql.catalog.lakehouse",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type",      "rest")
    .config("spark.sql.catalog.lakehouse.uri",       "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    # S3FileIO writes data files directly to MinIO
    .config("spark.sql.catalog.lakehouse.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint",          "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id",     os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}   catalog: lakehouse")

# Create your database once
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.taxi")

Spark 4.0.1   catalog: lakehouse


DataFrame[]

In [3]:
BOOTSTRAP = "kafka:9092"
TOPIC = "taxi-trips"

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

# Bronze

In [4]:
BRONZE_TABLE = "lakehouse.taxi.bronze_raw_events"
CHECKPOINT_PATH = "/home/jovyan/work/checkpoints/bronze_raw_events"

# Create a raw Bronze table that mirrors Kafka payload + metadata as-is.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
    key BINARY,
    value BINARY,
    topic STRING,
    partition INT,
    offset BIGINT,
    timestamp TIMESTAMP,
    timestampType INT
) USING iceberg
""")

bronze_query = (
    raw_stream.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .toTable(BRONZE_TABLE)
)

print(f"Bronze stream started: {bronze_query.id}")
print(f"Checkpoint path: {CHECKPOINT_PATH}")

Bronze stream started: 1bd77762-6990-4c6c-a8d1-333781f63654
Checkpoint path: /home/jovyan/work/checkpoints/bronze_raw_events


In [5]:
raw = spark.read.table("lakehouse.taxi.bronze_raw_events")

raw.createOrReplaceTempView("bronze")

spark.sql("SELECT count(*) FROM bronze").show()
spark.sql("SELECT * FROM bronze LIMIT 3").show()

CHECKPOINT_PATH = "/home/jovyan/work/checkpoints/bronze_raw_events"

+--------+
|count(1)|
+--------+
|    5869|
+--------+

+----+--------------------+----------+---------+------+--------------------+-------------+
| key|               value|     topic|partition|offset|           timestamp|timestampType|
+----+--------------------+----------+---------+------+--------------------+-------------+
|[31]|[7B 22 56 65 6E 6...|taxi-trips|        0|  1227|2026-03-30 18:36:...|            0|
|[31]|[7B 22 56 65 6E 6...|taxi-trips|        0|  1228|2026-03-30 18:36:...|            0|
|[32]|[7B 22 56 65 6E 6...|taxi-trips|        2|  4488|2026-03-30 18:36:...|            0|
+----+--------------------+----------+---------+------+--------------------+-------------+



# Silver

In [6]:
from pyspark.sql.types import (
    StructType, StructField,
    LongType, DoubleType, StringType, IntegerType, TimestampType
)

## Define schema

In [7]:
TRIP_SCHEMA = StructType([
    StructField("VendorID",              LongType(),   True),
    StructField("tpep_pickup_datetime",  StringType(), True),
    StructField("tpep_dropoff_datetime", StringType(), True),
    StructField("passenger_count",       DoubleType(), True),
    StructField("trip_distance",         DoubleType(), True),
    StructField("RatecodeID",            DoubleType(), True),
    StructField("store_and_fwd_flag",    StringType(), True),
    StructField("PULocationID",          LongType(),   True),
    StructField("DOLocationID",          LongType(),   True),
    StructField("payment_type",          LongType(),   True),
    StructField("fare_amount",           DoubleType(), True),
    StructField("extra",                 DoubleType(), True),
    StructField("mta_tax",               DoubleType(), True),
    StructField("tip_amount",            DoubleType(), True),
    StructField("tolls_amount",          DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount",          DoubleType(), True),
    StructField("congestion_surcharge",  DoubleType(), True),
    StructField("Airport_fee",           DoubleType(), True),
    StructField("cbd_congestion_fee",    DoubleType(), True),
])

### 2. Read bronze  batch

In [8]:
silver_batch = (
    spark.read
    .format("iceberg")
    .table("lakehouse.taxi.bronze_raw_events")
)

In [9]:

def parse(silver_read):
    # ---------------------------------------------------------------------------
    # 3. R1  Parse JSON payload
    # ---------------------------------------------------------------------------
    parsed = (
        silver_batch
        .withColumn("value_str", F.col("value").cast("string"))
        .withColumn("trip",      F.from_json(F.col("value_str"), TRIP_SCHEMA))
        .select("trip.*",
                # keep Kafka metadata
                F.col("timestamp").alias("kafka_ingest_ts"))
    )

    # ---------------------------------------------------------------------------
    # 4. R2  Cast timestamp strings ΓåÆ TimestampType
    #    Input format: "2025-01-01T12:45:51" (ISO-8601 without timezone)
    # ---------------------------------------------------------------------------
    parsed = (
        parsed
        .withColumn("pickup_datetime",
                    F.to_timestamp("tpep_pickup_datetime",  "yyyy-MM-dd'T'HH:mm:ss"))
        .withColumn("dropoff_datetime",
                    F.to_timestamp("tpep_dropoff_datetime", "yyyy-MM-dd'T'HH:mm:ss"))
        .drop("tpep_pickup_datetime", "tpep_dropoff_datetime")
    )
    return parsed

# Clean

In [10]:
def clean(parsed):
    cleaned = (
        parsed

        # R3  passenger_count: float ΓåÆ int; must be 1ΓÇô9
        .withColumn("passenger_count",
                    F.when(
                        F.col("passenger_count").cast(
                            IntegerType()).between(1, 9),
                        F.col("passenger_count").cast(IntegerType())
                    ).otherwise(F.lit(None).cast(IntegerType())))

        # R4  trip_distance must be positive
        .withColumn("trip_distance",
                    F.when(F.col("trip_distance") > 0, F.col("trip_distance"))
                     .otherwise(None))

        # R5  RatecodeID valid domain: 1=Standard, 2=JFK, 3=Newark,
        #        4=Nassau/Westchester, 5=Negotiated, 6=Group ride
        .withColumn("RatecodeID",
                    F.when(F.col("RatecodeID").cast(IntegerType()).between(1, 6),
                           F.col("RatecodeID").cast(IntegerType()))
                    .otherwise(None))

        # R6  payment_type valid domain: 1=Credit, 2=Cash, 3=No charge,
        #        4=Dispute, 5=Unknown, 6=Voided
        .withColumn("payment_type",
                    F.when(F.col("payment_type").between(
                        1, 6), F.col("payment_type"))
                     .otherwise(None))

        # R7  fare_amount cannot be negative
        .withColumn("fare_amount",
                    F.when(F.col("fare_amount") >= 0, F.col("fare_amount"))
                     .otherwise(None))

        # R8  total_amount cannot be negative
        .withColumn("total_amount",
                    F.when(F.col("total_amount") >= 0, F.col("total_amount"))
                     .otherwise(None))

        # R9  dropoff must be strictly after pickup
        .withColumn("is_valid_window",
                    F.col("dropoff_datetime") > F.col("pickup_datetime"))
        .filter(F.col("is_valid_window"))
        .drop("is_valid_window")

        # Derived: trip duration in minutes (convenient for analytics)
        .withColumn("trip_duration_min",
                    F.round(
                        (F.unix_timestamp("dropoff_datetime")
                         - F.unix_timestamp("pickup_datetime")) / 60.0,
                        2))

        # Partition column for Iceberg  date extracted from pickup
        .withColumn("pickup_date", F.to_date("pickup_datetime"))
    )
    return cleaned

# Deduplicate

In [11]:
def dedup(cleaned):
    DEDUP_KEY = ["VendorID", "pickup_datetime", "dropoff_datetime",
                 "PULocationID", "DOLocationID"]
    return cleaned.dropDuplicates(DEDUP_KEY)

# Zone enrichment (pickup + dropoff)

In [12]:
def enrich(deduped):
    zones = spark.read.parquet(
        "/home/jovyan/project/data/taxi_zone_lookup.parquet").alias("z")

    enriched = (
        deduped.alias("t")

        # Pickup zone
        .join(
            zones.select(
                F.col("LocationID").alias("pu_loc_id"),
                F.col("Zone").alias("pickup_zone"),
                F.col("Borough").alias("pickup_borough"),
                F.col("service_zone").alias("pickup_service_zone"),
            ),
            F.col("t.PULocationID") == F.col("pu_loc_id"),
            how="left"
        )
        .drop("pu_loc_id")

        # Dropoff zone
        .join(
            zones.select(
                F.col("LocationID").alias("do_loc_id"),
                F.col("Zone").alias("dropoff_zone"),
                F.col("Borough").alias("dropoff_borough"),
                F.col("service_zone").alias("dropoff_service_zone"),
            ),
            F.col("t.DOLocationID") == F.col("do_loc_id"),
            how="left"
        )
        .drop("do_loc_id")
    )
    return enriched

# Final column order

In [13]:
def order(enriched):
    silver = enriched.select(
        # identifiers
        "VendorID",
        # time
        "pickup_datetime", "dropoff_datetime", "trip_duration_min", "pickup_date",
        # geography
        "PULocationID", "pickup_zone",  "pickup_borough",
        "DOLocationID", "dropoff_zone", "dropoff_borough",
        # trip facts
        "passenger_count", "trip_distance", "RatecodeID", "store_and_fwd_flag",
        # fares
        "fare_amount", "extra", "mta_tax", "tip_amount",
        "tolls_amount", "improvement_surcharge",
        "congestion_surcharge", "Airport_fee", "cbd_congestion_fee",
        "total_amount", "payment_type",
        # lineage
        "kafka_ingest_ts",
    )
    return silver


def transform(silver_stream):

    return order(enrich(dedup(clean(parse(silver_stream)))))

In [14]:
SILVER_CHECKPOINT = "/home/jovyan/work/checkpoints/bronze_raw_events"

silver_schema = transform(silver_batch).schema
table_name = "lakehouse.taxi.silver_trips"

if not spark.catalog.tableExists(table_name):
    print(f"Creating {table_name} for the first time...")

    spark.catalog.createTable(
        table_name, schema=silver_schema, source="iceberg")

    # Apply partitioning and optimizations via SQL
    spark.sql(f"ALTER TABLE {table_name} ADD PARTITION FIELD pickup_date")
    spark.sql(
        f"ALTER TABLE {table_name} SET TBLPROPERTIES ('write.format.default'='parquet', 'write.parquet.compression-codec'='zstd')")
    print("Table initialized.")
else:
    print(f"{table_name} already exists. Skipping initialization.")

lakehouse.taxi.silver_trips already exists. Skipping initialization.


In [15]:
silver_batch_transformed = transform(silver_batch)

(
    silver_batch_transformed
    .writeTo("lakehouse.taxi.silver_trips")
    .overwritePartitions()
)

spark.read.table("lakehouse.taxi.silver_trips").count()

4054

# Gold

Reading batch from the silver layer

In [16]:
gold_batch = (
    spark.read
    .format("iceberg")
    .table("lakehouse.taxi.silver_trips")
)


def gold_writer(df, table_name):
    return (
        df.writeTo(table_name)
        .using("iceberg")
        .tableProperty("write.format.default", "parquet")
        .tableProperty("write.parquet.compression-codec", "zstd")
    )

Aggregation 1: Trip counts per pickup hour

In [17]:
def get_trip_counts_per_pickup_hour(batch):
    pick_up_hour_df = batch.withColumn(
        "pickup_hour", F.date_trunc("hour", F.col("pickup_datetime"))
    )

    df = pick_up_hour_df.groupBy(
        "pickup_date", "pickup_hour"
    ).count().withColumnRenamed("count", "trip_count")

    return df

In [18]:
gold_trips_per_hour_table = "lakehouse.taxi.gold_trips_per_hour"

gold_trips_per_hour_df = get_trip_counts_per_pickup_hour(gold_batch)

if not spark.catalog.tableExists(gold_trips_per_hour_table):
    # First time: define layout + load data in one step
    gold_writer(gold_trips_per_hour_df, gold_trips_per_hour_table).partitionedBy(
        "pickup_date").create()
else:
    # Every batch: replace only partitions present in gold_df
    gold_writer(gold_trips_per_hour_df,
                gold_trips_per_hour_table).overwritePartitions()

In [19]:
gold_trips_per_hour_after_ins = spark.table(gold_trips_per_hour_table)
gold_trips_per_hour_after_ins.show(20)

+-----------+-------------------+----------+
|pickup_date|        pickup_hour|trip_count|
+-----------+-------------------+----------+
| 2024-12-31|2024-12-31 23:00:00|         9|
| 2025-01-01|2025-01-01 00:00:00|      4041|
| 2025-01-01|2025-01-01 01:00:00|         4|
+-----------+-------------------+----------+



Aggregation 2: Average fare per pickup zone

In [20]:
def get_average_fare_per_pickup_zone(batch):
    df = batch.groupBy(
        "pickup_date", "PULocationID", "pickup_borough", "pickup_zone"
    ).agg(
        F.round(F.avg("fare_amount"), 2).alias("average_fare")
    ).withColumnRenamed(
        "PULocationID", "pickup_location_id"
    )

    return df

In [21]:
gold_average_fare_per_zone_table = "lakehouse.taxi.gold_average_fare_per_zone"

gold_average_fare_per_zone_df = get_average_fare_per_pickup_zone(gold_batch)

if not spark.catalog.tableExists(gold_average_fare_per_zone_table):
    gold_writer(gold_average_fare_per_zone_df,
                gold_average_fare_per_zone_table).partitionedBy("pickup_date").create()
else:
    gold_writer(gold_average_fare_per_zone_df,
                gold_average_fare_per_zone_table).overwritePartitions()

In [22]:
gold_average_fare_per_zone_after_ins = spark.table(
    "lakehouse.taxi.gold_average_fare_per_zone")
gold_average_fare_per_zone_after_ins.show(20)

+-----------+------------------+--------------+--------------------+------------+
|pickup_date|pickup_location_id|pickup_borough|         pickup_zone|average_fare|
+-----------+------------------+--------------+--------------------+------------+
| 2024-12-31|               114|     Manhattan|Greenwich Village...|        34.5|
| 2024-12-31|                68|     Manhattan|        East Chelsea|        14.9|
| 2024-12-31|               142|     Manhattan| Lincoln Square East|        12.1|
| 2024-12-31|                56|        Queens|              Corona|         7.9|
| 2024-12-31|                43|     Manhattan|        Central Park|        10.7|
| 2024-12-31|               246|     Manhattan|West Chelsea/Huds...|        16.3|
| 2024-12-31|               229|     Manhattan|Sutton Place/Turt...|       13.15|
| 2024-12-31|               170|     Manhattan|         Murray Hill|        10.0|
| 2025-01-01|                 4|     Manhattan|       Alphabet City|       13.19|
| 2025-01-01|   

Aggregation 3: Revenue per pick up zone

In [23]:
def get_revenue_per_pickup_zone(batch):
    df = batch.groupBy(
        "pickup_date", "PULocationID", "pickup_borough", "pickup_zone"
    ).agg(
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
    ).withColumnRenamed("PULocationID", "pickup_location_id")
    return df

In [24]:
gold_revenue_per_zone_table = "lakehouse.taxi.gold_revenue_per_zone"

gold_revenue_per_zone_df = get_revenue_per_pickup_zone(gold_batch)

if not spark.catalog.tableExists(gold_revenue_per_zone_table):
    gold_writer(gold_revenue_per_zone_df, gold_revenue_per_zone_table).partitionedBy(
        "pickup_date").create()
else:
    gold_writer(gold_revenue_per_zone_df,
                gold_revenue_per_zone_table).overwritePartitions()

In [25]:
gold_revenue_per_zone_after_ins = spark.table(
    "lakehouse.taxi.gold_revenue_per_zone")
gold_revenue_per_zone_after_ins.show(20)

+-----------+------------------+--------------+--------------------+-------------+
|pickup_date|pickup_location_id|pickup_borough|         pickup_zone|total_revenue|
+-----------+------------------+--------------+--------------------+-------------+
| 2024-12-31|               114|     Manhattan|Greenwich Village...|         47.4|
| 2024-12-31|                68|     Manhattan|        East Chelsea|        23.88|
| 2024-12-31|               142|     Manhattan| Lincoln Square East|        20.52|
| 2024-12-31|                56|        Queens|              Corona|         10.4|
| 2024-12-31|                43|     Manhattan|        Central Park|         15.7|
| 2024-12-31|               246|     Manhattan|West Chelsea/Huds...|        26.62|
| 2024-12-31|               229|     Manhattan|Sutton Place/Turt...|        38.13|
| 2024-12-31|               170|     Manhattan|         Murray Hill|         18.0|
| 2025-01-01|                 4|     Manhattan|       Alphabet City|       179.29|
| 20

# Bronze trigger benchmark

This benchmark replays the same bounded Kafka input three times with different Structured Streaming trigger intervals.

Configuration used in the run:

- Input: first 360 rows from `yellow_tripdata_2025-01.parquet`
- Send rate: 10 events per second
- Triggers: `5 seconds`, `30 seconds`, `1 minute`

The goal is to measure how trigger frequency changes Bronze-layer file counts, batch counts, and end-to-end latency.

In [10]:
import importlib.util
import json
import subprocess
import sys
import time
import uuid


def ensure_package(pkg, import_name=None):
    if importlib.util.find_spec(import_name or pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


ensure_package("kafka-python", "kafka")

from kafka import KafkaProducer
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError


BENCHMARK_TRIGGERS = [
    {"trigger": "5 seconds", "slug": "5s"},
    {"trigger": "30 seconds", "slug": "30s"},
    {"trigger": "1 minute", "slug": "1m"},
]
BENCHMARK_ROWS = 360
BENCHMARK_RATE = 10.0
BENCHMARK_BOOTSTRAP = "kafka:9092"
BENCHMARK_DATA_PATH = "/home/jovyan/project/data/yellow_tripdata_2025-01.parquet"
BENCHMARK_CHECKPOINT_ROOT = "/home/jovyan/project/work/checkpoints/bronze_trigger_benchmark"


def load_benchmark_messages(rows=BENCHMARK_ROWS, data_path=BENCHMARK_DATA_PATH):
    sample = spark.read.parquet(data_path).limit(rows)
    payload = F.to_json(F.struct(*[F.col(column) for column in sample.columns]))
    batch = sample.select(
        F.coalesce(F.col("VendorID").cast("string"), F.lit("")).alias("message_key"),
        payload.alias("message_value"),
    ).collect()
    return [(row.message_key, row.message_value) for row in batch]


def create_topic(bootstrap, topic):
    admin = KafkaAdminClient(bootstrap_servers=bootstrap)
    try:
        admin.create_topics([NewTopic(name=topic, num_partitions=3, replication_factor=1)])
    except TopicAlreadyExistsError:
        pass
    finally:
        admin.close()


def create_bronze_table(table_name):
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
        key BINARY,
        value BINARY,
        topic STRING,
        partition INT,
        offset BIGINT,
        timestamp TIMESTAMP,
        timestampType INT
    ) USING iceberg
    """)


def send_messages(messages, bootstrap, topic, rate):
    producer = KafkaProducer(bootstrap_servers=bootstrap, acks=1, retries=3)
    interval = 1.0 / rate
    try:
        for key, value in messages:
            producer.send(topic, key=key.encode(), value=value.encode())
            time.sleep(interval)
    finally:
        producer.flush()
        producer.close()


def wait_for_rows(query, table_name, expected_rows, timeout_seconds):
    deadline = time.monotonic() + timeout_seconds
    while time.monotonic() < deadline:
        query_exception = query.exception()
        if query_exception is not None:
            raise RuntimeError(query_exception)

        spark.catalog.refreshTable(table_name)
        row_count = spark.read.table(table_name).count()
        if row_count >= expected_rows:
            return row_count
        time.sleep(1.0)
    raise TimeoutError(f"Timed out waiting for {expected_rows} rows in {table_name}.")


def count_output_files(table_name):
    spark.catalog.refreshTable(table_name)
    row = spark.sql(f"SELECT count(*) AS file_count FROM {table_name}.files").first()
    return int(row.file_count)


def count_data_batches(table_name):
    spark.catalog.refreshTable(table_name)
    row = spark.sql(f"""
    SELECT count(*) AS batch_count
    FROM {table_name}.snapshots
    WHERE CAST(summary['added-records'] AS BIGINT) > 0
    """).first()
    return int(row.batch_count)


def run_bronze_trigger_benchmark(trigger, slug, messages, rate=BENCHMARK_RATE, bootstrap=BENCHMARK_BOOTSTRAP, run_id=None):
    run_id = run_id or uuid.uuid4().hex[:8]
    topic = f"taxi-trips-benchmark-{slug}-{run_id}"
    table_name = f"lakehouse.taxi.bronze_benchmark_{slug}_{run_id}"
    checkpoint_path = f"{BENCHMARK_CHECKPOINT_ROOT}/{slug}_{run_id}"

    create_topic(bootstrap, topic)
    create_bronze_table(table_name)

    benchmark_stream = (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", bootstrap)
        .option("subscribe", topic)
        .option("startingOffsets", "earliest")
        .load()
    )

    query = (
        benchmark_stream.writeStream
        .format("iceberg")
        .outputMode("append")
        .trigger(processingTime=trigger)
        .option("checkpointLocation", checkpoint_path)
        .toTable(table_name)
    )

    expected_rows = len(messages)
    trigger_seconds = 60 if trigger == "1 minute" else int(trigger.split()[0])
    timeout_seconds = max((expected_rows / rate) + (trigger_seconds * 3), 180.0)

    time.sleep(1.0)
    start_time = time.monotonic()
    try:
        send_messages(messages, bootstrap, topic, rate)
        rows_written = wait_for_rows(query, table_name, expected_rows, timeout_seconds)
        end_time = time.monotonic()
    finally:
        query.stop()
        query.awaitTermination(30)

    return {
        "trigger": trigger,
        "topic": topic,
        "table": table_name,
        "rows": rows_written,
        "batches": count_data_batches(table_name),
        "output_files": count_output_files(table_name),
        "end_to_end_seconds": round(end_time - start_time, 2),
    }

In [11]:
benchmark_messages = load_benchmark_messages()
benchmark_run_id = uuid.uuid4().hex[:8]

benchmark_results = []
for benchmark in BENCHMARK_TRIGGERS:
    benchmark_results.append(
        run_bronze_trigger_benchmark(
            trigger=benchmark["trigger"],
            slug=benchmark["slug"],
            messages=benchmark_messages,
            run_id=benchmark_run_id,
        )
    )

print(json.dumps(benchmark_results, indent=2))

[
  {
    "trigger": "5 seconds",
    "topic": "taxi-trips-benchmark-5s-92c891e0",
    "table": "lakehouse.taxi.bronze_benchmark_5s_92c891e0",
    "rows": 360,
    "batches": 8,
    "output_files": 16,
    "end_to_end_seconds": 38.33
  },
  {
    "trigger": "30 seconds",
    "topic": "taxi-trips-benchmark-30s-92c891e0",
    "table": "lakehouse.taxi.bronze_benchmark_30s_92c891e0",
    "rows": 360,
    "batches": 2,
    "output_files": 4,
    "end_to_end_seconds": 48.98
  },
  {
    "trigger": "1 minute",
    "topic": "taxi-trips-benchmark-1m-92c891e0",
    "table": "lakehouse.taxi.bronze_benchmark_1m_92c891e0",
    "rows": 360,
    "batches": 2,
    "output_files": 4,
    "end_to_end_seconds": 88.27
  }
]
